In [1]:

from google.colab import drive
import os
import subprocess
from tqdm import tqdm

In [2]:
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ---------------------------------------------------------
# Path configuration
# ---------------------------------------------------------
INPUT_ROOT = "/content/drive/Shareddrives/Sandler and Kaminka/DICOM/Bladder 13.11.25"# Source folders
OUTPUT_ROOT = "/content/drive/MyDrive/Bladder organized"  # Output folders
SCRIPT_PATH = "/content/drive/Shareddrives/Sandler and Kaminka/code/sort_sectra.py"                # Your original script

# ---------------------------------------------------------
# Validate paths before running anything
# ---------------------------------------------------------

def validate_paths():
    errors = []

    # Check input folder
    if not os.path.exists(INPUT_ROOT):
        errors.append(f"❌ ERROR: INPUT_ROOT does not exist: {INPUT_ROOT}")

    # Check script file
    if not os.path.exists(SCRIPT_PATH):
        errors.append(f"❌ ERROR: SCRIPT_PATH not found: {SCRIPT_PATH}\n"
                      f"Make sure you uploaded sort_sectra.py to /content")

    # Create output folder if missing
    if not os.path.exists(OUTPUT_ROOT):
        print(f"⚠️ OUTPUT_ROOT not found, creating it: {OUTPUT_ROOT}")
        os.makedirs(OUTPUT_ROOT, exist_ok=True)

    # If any errors were found, stop execution
    if errors:
        print("\n".join(errors))
        raise SystemExit("⛔ Stopping execution due to missing paths.")

    print("✅ All paths validated successfully.")
    print(f"   INPUT_ROOT:  {INPUT_ROOT}")
    print(f"   OUTPUT_ROOT: {OUTPUT_ROOT}")
    print(f"   SCRIPT_PATH: {SCRIPT_PATH}")

# Run validation
validate_paths()

✅ All paths validated successfully.
   INPUT_ROOT:  /content/drive/Shareddrives/Sandler and Kaminka/DICOM/Bladder 13.11.25
   OUTPUT_ROOT: /content/drive/MyDrive/Bladder organized
   SCRIPT_PATH: /content/drive/Shareddrives/Sandler and Kaminka/code/sort_sectra.py


In [4]:

# ---------------------------------------------------------
# Check if a folder contains a Sectra XML file
# ---------------------------------------------------------
def has_xml(folder):
    xml1 = os.path.join(folder, "content.xml")
    xml2 = os.path.join(folder, "SECTRA", "content.xml")
    return os.path.exists(xml1) or os.path.exists(xml2)

# ---------------------------------------------------------
# Run the original script on a single folder
# ---------------------------------------------------------
def process_folder(src, dst):
    os.makedirs(dst, exist_ok=True)

    cmd = [
        "python3",
        SCRIPT_PATH,
        src,
        dst,
        "--debug"
    ]

    try:
        subprocess.run(cmd, check=True)
        return True
    except subprocess.CalledProcessError:
        return False


In [5]:

# ---------------------------------------------------------
# Collect all patient folders
# ---------------------------------------------------------
patient_folders = [
    f for f in os.listdir(INPUT_ROOT)
    if os.path.isdir(os.path.join(INPUT_ROOT, f))
]

print(f"Found {len(patient_folders)} folders in source directory")

Found 66 folders in source directory


In [6]:
#---------------------------------------------------------
#  Choose: test mode (3 folders) or full run
# ---------------------------------------------------------
TEST_MODE = True   # ← Change to False for full processing

if TEST_MODE:
    folders_to_run = patient_folders[:3]
    print("🔍 TEST MODE ENABLED — processing only 3 folders")
else:
    folders_to_run = patient_folders
    print("🚀 FULL RUN MODE — processing all folders")

🔍 TEST MODE ENABLED — processing only 3 folders


In [7]:
!pip install pydicom

In [8]:
import xml.etree.ElementTree as ET
import re
import argparse
from datetime import datetime
import pydicom
from pydicom.uid import generate_uid


In [14]:
# ---------------------------------------------------------
# Create master log file
# ---------------------------------------------------------
log_file_path = os.path.join(OUTPUT_ROOT, "master_log.txt")
log_file = open(log_file_path, "w", encoding="utf-8")

def log(msg):
    """Print to screen AND write to log file."""
    print(msg)
    log_file.write(msg + "\n")

# ---------------------------------------------------------
# Process folders with progress bar
# ---------------------------------------------------------
for folder in tqdm(folders_to_run, desc="Processing folders"):
    src = os.path.join(INPUT_ROOT, folder)
    dst = os.path.join(OUTPUT_ROOT, folder)

    log(f"\n=== Processing {folder} ===")

    # Skip folders already processed
    if os.path.exists(dst) and os.listdir(dst):
        log("Skipped (already processed)")
        continue

    # Run the processing
    ok = process_folder(src, dst)

    if ok:
        log("✅ Success")
    else:
        log("❌ Failed")

log_file.close()

print("\n🎉 DONE! Master log saved at:")
print(log_file_path)

Processing folders:   0%|          | 0/3 [00:00<?, ?it/s]


=== Processing 31504427 ===


Processing folders:  33%|███▎      | 1/3 [15:09<30:18, 909.46s/it]

✅ Success

=== Processing 3129058 ===


Processing folders:  67%|██████▋   | 2/3 [31:16<15:43, 943.61s/it]

✅ Success

=== Processing 31574629 ===


Processing folders: 100%|██████████| 3/3 [48:11<00:00, 963.82s/it]

✅ Success

🎉 DONE! Master log saved at:
/content/drive/MyDrive/Bladder organized/master_log.txt


Run this cell in order to see sort_sectra.py prosses on one file

In [13]:
!python3 "/content/drive/Shareddrives/Sandler and Kaminka/code/sort_sectra.py" \
    "/content/drive/Shareddrives/Sandler and Kaminka/DICOM/Bladder 13.11.25/30977820" \
    "/content/drive/MyDrive/Bladder organized/30977820" \
    --debug

[INFO] Debug mode: True
[INFO] Log file: /content/drive/MyDrive/Bladder organized/30977820/sort_log_20260325_205418.txt
[INFO] Starting sort for patient folder: /content/drive/Shareddrives/Sandler and Kaminka/DICOM/Bladder 13.11.25/30977820
[INFO] Found XML: /content/drive/Shareddrives/Sandler and Kaminka/DICOM/Bladder 13.11.25/30977820/SECTRA/CONTENT.XML
[INFO] Found 14 series in XML
[INFO] Generated Study UID: 1.2.826.0.1.3680043.8.498.13768452247359826496484023607251068647
[INFO] 
===== PET ↔ CT MATCHING =====
[INFO] PT_ONE_BED_HD_S79                   → CT_CT_2.5_MM_S405
[INFO] PT_ONE_BED_NAC_S79                  → CT_CTAC_3.75_THICK_S311
[INFO] PT_ONE_BED_QC_S79                   → CT_CT_2.5_MM_S405
[INFO] PT_WB_NAC_S311                      → CT_CTAC_3.75_THICK_S311
[INFO] PT_WB_QC_350_S311                   → CT_CT_WB_STANDARD_S311
[INFO] PT_WB_VPHD-S_MAC_S311               → CT_CTAC_3.75_THICK_S311
[INFO] 
===== APPLYING PET ↔ CT DICOM REFERENCES =====
[WARNING] Missing PET or 